# Flood Impact Tags — Transition Profiles & Geographic Distribution

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import rplot

plt.style.use('ryan')


TAGS  = '/home/ryan/data/flood_hazard/metadata/flood_impact_tags.parquet'
SITES = '/home/ryan/data/flood_hazard/metadata/site_info.parquet'

LEVELS = ['action', 'flood', 'moderate', 'major']
TAG_NAMES = [
    'road_flooded', 'road_closed', 'bridge_threatened',
    'homes_threatened', 'homes_flooded', 'businesses_flooded',
    'agricultural', 'natural_lowland', 'recreational',
    'evacuation', 'utilities', 'widespread',
]

tags  = pd.read_parquet(TAGS)
sites = pd.read_parquet(SITES)[['site_no', 'latitude', 'longitude', 'state_cd']]
tags  = tags.merge(sites, on='site_no', how='left')
tags['level'] = pd.Categorical(tags['level'], categories=LEVELS, ordered=True)
print(f'Tags: {len(tags):,} rows | Sites with coords: {tags.latitude.notna().sum():,}')

## 1. Tag transition profiles

How does each tag's prevalence change across the four severity levels, and at which level does it first appear for individual sites?

In [ ]:
# 1a. Prevalence curves
prev = (
    tags.groupby('level', observed=True)[TAG_NAMES]
        .mean()
        .mul(100)
        .loc[LEVELS]
)

INFRA   = ['road_flooded', 'road_closed', 'bridge_threatened']
RESID   = ['homes_threatened', 'homes_flooded', 'businesses_flooded']
NATURAL = ['natural_lowland', 'agricultural', 'recreational']
OTHER   = ['evacuation', 'utilities', 'widespread']
palette = {
    **{t: '#1f77b4' for t in INFRA},
    **{t: '#d62728' for t in RESID},
    **{t: '#2ca02c' for t in NATURAL},
    **{t: '#9467bd' for t in OTHER},
}
solid = {'road_flooded', 'homes_flooded', 'natural_lowland', 'widespread'}

fig, ax = plt.subplots(figsize=(10, 5), dpi=300)
for tag in TAG_NAMES:
    ls = '-' if tag in solid else '--'
    lw = 2.2 if tag in solid else 1.2
    ax.plot(LEVELS, prev[tag], marker='o', ms=5,
            color=palette[tag], ls=ls, lw=lw, label=tag)
ax.set_xlabel('Severity level')
ax.set_ylabel('% of descriptions with tag')
ax.set_title('Tag prevalence across severity levels')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8, frameon=False)
ax.yaxis.grid(True, alpha=0.3)
ax.set_ylim(0, None)
plt.tight_layout()
plt.show()

In [ ]:
# 1b. Heatmap
fig, ax = plt.subplots(figsize=(7, 5), dpi=300)
im = ax.imshow(prev.T.values.astype(float), aspect='auto', cmap=rplot.SEQ_HALINE.reversed(), vmin=0, vmax=60)
ax.set_xticks(range(4))
ax.set_xticklabels(LEVELS, fontsize=11)
ax.set_yticks(range(len(TAG_NAMES)))
ax.set_yticklabels(TAG_NAMES, fontsize=9)
plt.colorbar(im, ax=ax, label='% of descriptions')
for i, tag in enumerate(TAG_NAMES):
    for j, lvl in enumerate(LEVELS):
        val = float(prev.loc[lvl, tag])
        ax.text(j, i, f'{val:.0f}', ha='center', va='center',
                fontsize=7, color='white' if val > 35 else 'black')
        

ax.grid(False)
ax.set_title('Tag prevalence (%) by severity level')
plt.tight_layout()
plt.show()

In [ ]:
# 1c. First-appearance level per site
first_on = {}
for tag in TAG_NAMES:
    rows = []
    for site, grp in tags.sort_values('level').groupby('site_no'):
        on = grp.loc[grp[tag] == True, 'level']
        if len(on) > 0:
            rows.append(str(on.iloc[0]))
    counts = pd.Series(rows).value_counts().reindex(LEVELS, fill_value=0)
    first_on[tag] = counts

first_df  = pd.DataFrame(first_on, index=LEVELS).T
first_pct = first_df.div(first_df.sum(axis=1), axis=0).mul(100)

# Sort by % first appearing at action level
first_pct = first_pct.sort_values('action', ascending=True)

fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
bottom = np.zeros(len(first_pct))
colors = ['#FFFF00', '#FF9900', '#FF0000', '#CC33FF']
for lvl, color in zip(LEVELS, colors):
    vals = first_pct[lvl].values
    ax.barh(first_pct.index, vals, left=bottom, color=color, label=lvl, edgecolor='none', alpha=0.7)
    bottom += vals
ax.set_xlabel('% of tagged sites where tag first appears at this level')
ax.set_title('At which severity level does each tag first appear?')
ax.legend(title='First level', bbox_to_anchor=(1.01, 1), loc='upper left', frameon=False)
ax.set_xlim(0, 100)
# ax.xaxis.grid(True, alpha=0.3)
ax.grid(False)
plt.tight_layout()
plt.show()

## 2. Geographic distribution

Spatial pattern of tag prevalence across the continental US.

In [ ]:
LON_MIN, LON_MAX = -125, -66
LAT_MIN, LAT_MAX =   24,  50

def site_level(level):
    return tags[tags['level'] == level].dropna(subset=['latitude', 'longitude'])

In [ ]:
# # 2a. Grid of maps: tag presence at major stage
# sub = site_level('major')
# ncols = 4
# nrows = -(-len(TAG_NAMES) // ncols)

# fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3.2))
# axes = axes.flatten()
# for ax, tag in zip(axes, TAG_NAMES):
#     neg = sub[sub[tag] == False]
#     pos = sub[sub[tag] == True]
#     ax.scatter(neg.longitude, neg.latitude, s=2, c='#cccccc', alpha=0.3, linewidths=0)
#     ax.scatter(pos.longitude, pos.latitude, s=5, c='#d62728', alpha=0.7, linewidths=0)
#     ax.set_xlim(LON_MIN, LON_MAX)
#     ax.set_ylim(LAT_MIN, LAT_MAX)
#     ax.set_title(tag, fontsize=9)
#     ax.set_xticks([]); ax.set_yticks([])
#     pct = 100 * len(pos) / max(len(sub), 1)
#     ax.text(0.02, 0.04, f'{pct:.0f}%', transform=ax.transAxes,
#             fontsize=8, color='#d62728')
# for ax in axes[len(TAG_NAMES):]:
#     ax.set_visible(False)
# fig.suptitle('Tag presence at MAJOR stage  (red = tagged, % shown)', fontsize=13)
# plt.tight_layout()
# plt.show()

In [ ]:
# 2b. Cumulative impact load by site
load = (
    tags.dropna(subset=['latitude', 'longitude'])
        .groupby('site_no')
        .agg(total_tags=(TAG_NAMES[0], lambda x: 0))
)
# sum all tag columns across all levels per site
load = (
    tags.dropna(subset=['latitude', 'longitude'])
        .groupby('site_no')[TAG_NAMES].sum().sum(axis=1)
        .rename('total_tags')
)
coords = (
    tags.dropna(subset=['latitude', 'longitude'])
        .drop_duplicates('site_no')
        .set_index('site_no')[['latitude', 'longitude']]
)
load_df = coords.join(load)

fig, ax = plt.subplots(figsize=(12, 6))
sc = ax.scatter(
    load_df.longitude, load_df.latitude,
    c=load_df.total_tags, cmap=rplot.SEQ_MATTER.reversed(), s=12, alpha=0.7, linewidths=0,
    vmin=0, vmax=load_df.total_tags.quantile(0.97),
)
plt.colorbar(sc, ax=ax, label='Total tag activations across all severity levels')
ax.set_xlim(LON_MIN, LON_MAX); ax.set_ylim(LAT_MIN, LAT_MAX)
ax.set_title('Cumulative impact load by site')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
# # 2c. State-level summary at major stage
# major     = site_level('major')
# state_avg = major.groupby('state_cd')[TAG_NAMES].mean().mul(100)
# top_tags  = state_avg.mean().nlargest(5).index.tolist()
# plot_df   = state_avg[top_tags].dropna()

# fig, ax = plt.subplots(figsize=(14, 5))
# x, w = np.arange(len(plot_df)), 0.15
# for i, tag in enumerate(top_tags):
#     ax.bar(x + i * w, plot_df[tag], w, label=tag, edgecolor='none')
# ax.set_xticks(x + w * 2)
# ax.set_xticklabels(plot_df.index, fontsize=7, rotation=45, ha='right')
# ax.set_ylabel('% of sites with tag (major stage)')
# ax.set_title('State-level tag prevalence at major flood stage (top 5 tags)')
# ax.legend(fontsize=9, frameon=False)
# ax.yaxis.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()